In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/K2Debug/Financial-News-Sentiment-Analyzer-for-Economic-Forecasting.git"
PROJECT_ROOT = Path("/content/EF-02")
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        print("Workspace not found — cloning repo and installing dependencies...")
        !git clone {REPO_URL} {PROJECT_ROOT}
        !pip install -q python-dotenv groq openai bs4
        !pip install -q -r /content/EF-02/requirements.txt
        print("Workspace ready.")
    os.chdir(NOTEBOOKS_DIR)

## Section 1: Setup

In [2]:
import pandas as pd
import json
import time
import re
from groq import Groq
from dotenv import load_dotenv
from pathlib import Path
import os

# Prefer monorepo root ef02/.env; fall back to research/.env
load_dotenv(Path("..") / ".." / ".env")
load_dotenv(Path("..") / ".env")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.1-8b-instant"
BATCH_SIZE = 25
SLEEP_SEC  = 1.5

VALID_CATEGORIES = {
    "Forex", "Policy", "Banking", "Trade",
    "Agriculture", "Energy", "Transport", "Investment",
    "Markets", "Tourism", "Inflation"
}
VALID_SENTIMENTS = {"Positive", "Negative", "Neutral"}

## Section 2: Prompt

In [3]:
SYSTEM_PROMPT = """You are a financial news classifier for Tanzania.
You return ONLY a valid JSON array. No preamble, no explanation, no markdown fences.
Every object in the array must be complete and the array must be properly closed with ]."""


def build_prompt(batch, include_reason=False):
    lines = "\n".join(
        f"{i+1}. {item['headline']}"
        for i, item in enumerate(batch)
    )

    reason_field = ""
    if include_reason:
        reason_field = '\nAdd "reason": 4-6 words explaining your decision (debugging only).'

    return f"""You classify Tanzanian financial headlines. Output ONLY a valid JSON array.

Schema: [{{"pos":1,"relevant":true,"category":"Forex","sentiment":"Positive"}}, ...]{reason_field}

STEP 0 — RELEVANCE:
relevant true → Tanzania economy, finance, business, trade, banking, currency, energy, agriculture
relevant false → sports, crime, entertainment, foreign-only news, opinion/lifestyle; if false set category/sentiment null

STEP 1 — pick ONE category:
Forex(shilling,dollar,reserves) | Policy(BOT,IMF,budget,tax,debt,GDP,NBS,rates) | Banking(banks,loans,fintech,insurance) | Trade(imports,exports,tariffs,ports,AfCFTA) | Agriculture(crops,farming,food) | Energy(TANESCO,fuel,power) | Transport(SGR,rail,roads,logistics) | Investment(FDI,factories,PPP,crowdfunding) | Markets(DSE,CMSA,equity,bonds,turnover) | Tourism(hotels,arrivals) | Inflation(CPI,food/cement price spikes)

Disambiguation: DSE/CMSA→Markets | BOT rate/GDP→Policy | food/cement price spike→Inflation | hotels→Tourism

STEP 2 — SENTIMENT (economic outcome not tone):
Positive=growth,profit up,shilling firms,easing inflation,launches | Negative=decline,loss,miss target,weakening,rising inflation | Neutral=steady,guidelines,reviews,no clear direction
Falling inflation=Positive. Rising inflation=Negative.

Headlines:
{lines}"""

## Section 3: Test Run
Runs on a sample with `reason` off by default. Use `include_reason=True` only when inspecting individual decisions.

In [17]:
def classify_batch(batch, batch_num, total_batches, include_reason=False, max_retries=3):
    prompt = build_prompt(batch, include_reason=include_reason)
    raw = ""

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0,
                max_tokens=2500
            )

            raw   = response.choices[0].message.content.strip()
            clean = re.sub(r'```(?:json)?|```', '', raw).strip()
            parsed = json.loads(clean)

            if not isinstance(parsed, list):
                raise ValueError(f"Expected list, got {type(parsed)}")
            if len(parsed) != len(batch):
                raise ValueError(f"Got {len(parsed)} results for {len(batch)} headlines")

            for i, result in enumerate(parsed):
                result['id'] = batch[i]['id']
                result.pop('pos', None)

            print(f"  Batch {batch_num}/{total_batches} — OK ({len(parsed)} results)", end="\r", flush=True)
            return parsed

        except json.JSONDecodeError:
            tail = raw[-80:] if len(raw) > 80 else raw
            print(f"  Batch {batch_num} attempt {attempt}: JSON cut off. Ends: ...{repr(tail)}")
            time.sleep(2)

        except ValueError as e:
            print(f"  Batch {batch_num} attempt {attempt}: Validation error — {e}")
            time.sleep(2)

        except Exception as e:
            print(f"  Batch {batch_num} attempt {attempt}: API error — {e}")
            time.sleep(5)

    print(f"  Batch {batch_num}: FAILED after {max_retries} attempts. Filling with None.")
    return [
        {"id": item['id'], "relevant": None, "category": None, "sentiment": None}
        for item in batch
    ]


def run_classifier(input_csv, output_csv, batch_size=BATCH_SIZE,
                   test_size=None, include_reason=False):
    df = pd.read_csv(input_csv, engine='python', on_bad_lines='warn')
    print(f"Loaded: {len(df)} rows from {input_csv}")

    if test_size and test_size < len(df):
        df = df.sample(n=test_size, random_state=42).reset_index(drop=True)
        print(f"Sampled: {len(df)} rows for testing")
    else:
        df = df.reset_index(drop=True)
        print(f"Running on full dataset: {len(df)} rows")

    df['id'] = df.index

    rows    = df[['id', 'headline']].to_dict('records')
    batches = [rows[i:i+batch_size] for i in range(0, len(rows), batch_size)]
    total   = len(batches)
    print(f"\n{total} batches of up to {batch_size} headlines each")
    print(f"Estimated time: ~{total * SLEEP_SEC / 60:.1f} minutes\n")

    all_results = []
    for batch_num, batch in enumerate(batches, start=1):
        results = classify_batch(batch, batch_num, total, include_reason=include_reason)
        all_results.extend(results)
        if batch_num < total:
            time.sleep(SLEEP_SEC)

    df_results = pd.DataFrame(all_results)
    df_out     = df.merge(df_results, on='id', how='left').drop(columns=['id'])

    # Print summary
    total_rows = len(df_out)
    n_rel      = df_out['relevant'].sum()
    n_failed   = df_out['relevant'].isna().sum()

    print(f"\n{'='*50}")
    print(f"DONE")
    print(f"{'='*50}")
    print(f"  Total rows   : {total_rows}")
    print(f"  Relevant     : {int(n_rel)} ({n_rel/total_rows*100:.1f}%)")
    print(f"  Irrelevant   : {int(total_rows - n_rel)} ({(total_rows - n_rel)/total_rows*100:.1f}%)")
    if n_failed:
        print(f"  Failed rows  : {int(n_failed)} — run Section 5 retry cell to fix")

    print(f"\n  Category breakdown (relevant only):")
    for cat, count in df_out[df_out['relevant'] == True]['category'].value_counts().items():
        print(f"    {cat:<14} {count}")

    print(f"\n  Sentiment breakdown (relevant only):")
    for sent, count in df_out[df_out['relevant'] == True]['sentiment'].value_counts().items():
        print(f"    {sent:<14} {count}")

    df_out.to_csv(output_csv, index=False)
    print(f"\nSaved to {output_csv}")

    return df_out

In [ ]:
# Test run — 300 headlines, batch size 10, no reason field
df_test = run_classifier(
    input_csv      = "../data/processed/tz_headlines_clean.csv",
    output_csv     = "../data/processed/tz_headlines_test.csv",
    test_size      = 300,
    include_reason = False
)

In [30]:
# Inspect test results
df_test[df_test["relevant"]==False][["headline", "reason"]]

,headline,reason
33,Tanzania's Taifa Gas licensed to set up plant ...,no clear Tanzanian financial content
38,"EAC plans to host Africa, China – US business ...",no clear Tanzanian financial content
49,Fraud claims wipe $45 billion off India's Adan...,no clear Tanzanian financial content
63,Saudia celebrates Arabia’s 2034 FIFA WC Win wi...,no clear economic theme
135,Don’t be afraid of not taking a well-trodden p...,Don’t be afraid of not taking a well-trodden p...
140,China locks out Kenya from new debt relief deal,China locks out Kenya from new debt relief deal
141,State restore normalcy,State restore normalcy
142,The untold story of the making of Tanganyika -...,The untold story of the making of Tanganyika -...
147,Tasks ahead for new Vodacom Tanzania boss,Tasks ahead for new Vodacom Tanzania boss
149,What future has in store for local startups,What future has in store for local startups


## Section 4: Production Run
Full dataset, `reason` field OFF. Run only when happy with test results.

In [26]:
# Production run — full dataset, no reason field
df_labelled = run_classifier(
    input_csv      = "../data/processed/tz_headlines_clean.csv",
    output_csv     = "../data/processed/tz_headlines_labelled.csv",
    include_reason = False
)

Loaded: 5310 rows from ../data/processed/tz_headlines_clean.csv
Running on full dataset: 5310 rows

213 batches of up to 25 headlines each
Estimated time: ~5.3 minutes

  Batch 3 attempt 1: JSON cut off. Ends: ...'   "relevant": true,\n    "category": "Banking",\n    "sentiment": "Neutral"\n  ]\n]'
  Batch 3 attempt 2: JSON cut off. Ends: ...'   "relevant": true,\n    "category": "Banking",\n    "sentiment": "Neutral"\n  ]\n]'
  Batch 3 attempt 3: JSON cut off. Ends: ...'   "relevant": true,\n    "category": "Banking",\n    "sentiment": "Neutral"\n  ]\n]'
  Batch 3: FAILED after 3 attempts. Filling with None.
  Batch 6 attempt 1: Validation error — Got 1 results for 25 headlines
  Batch 6 attempt 2: Validation error — Got 1 results for 25 headlines
  Batch 6 attempt 3: Validation error — Got 1 results for 25 headlines
  Batch 6: FAILED after 3 attempts. Filling with None.
  Batch 21 attempt 1: JSON cut off. Ends: ...'s": 25,\n    "relevant": false,\n    "category": null,\n    "sentimen

## Section 5: Retry
Targets only rows where classification failed (relevant is NaN). Uses smaller batches to avoid token overflow.

In [28]:
LABELLED_CSV = "../data/processed/tz_headlines_labelled.csv"

df_full   = pd.read_csv(LABELLED_CSV)
df_failed = df_full[df_full['relevant'].isna()].copy().reset_index(drop=True)
print(f"Found {len(df_failed)} failed rows to retry")

if len(df_failed) == 0:
    print("Nothing to retry — all rows classified.")
else:
    df_failed['id'] = df_failed.index

    rows    = df_failed[['id', 'headline']].to_dict('records')
    batches = [rows[i:i+BATCH_SIZE] for i in range(0, len(rows), BATCH_SIZE)]
    total   = len(batches)
    print(f"{total} retry batches of up to {BATCH_SIZE} headlines each\n")

    retry_results = []
    for batch_num, batch in enumerate(batches, start=1):
        results = classify_batch(batch, batch_num, total, include_reason=False)
        retry_results.extend(results)
        if batch_num < total:
            time.sleep(SLEEP_SEC)

    # Patch results back into the full dataframe
    df_retry = pd.DataFrame(retry_results).drop(columns=['id'])
    failed_idx = df_full[df_full['relevant'].isna()].index
    df_full.loc[failed_idx, ['relevant', 'category', 'sentiment']] = \
        df_retry[['relevant', 'category', 'sentiment']].values

    still_failed = df_full['relevant'].isna().sum()
    print(f"\nRetry complete. Still failed: {int(still_failed)} rows")

    df_full.to_csv(LABELLED_CSV, index=False)
    print(f"Saved patched file to {LABELLED_CSV}")

Found 400 failed rows to retry
40 retry batches of up to 10 headlines each

  Batch 40/40 — OK (10 results)
Retry complete. Still failed: 0 rows
Saved patched file to ../data/processed/tz_headlines_labelled.csv


In [13]:
def summary(df):
    total_rows = len(df)
    n_rel      = df['relevant'].sum()
    n_failed   = df['relevant'].isna().sum()
    
    print(f"\n{'='*50}")
    print(f"DONE")
    print(f"{'='*50}")
    print(f"  Total rows   : {total_rows}")
    print(f"  Relevant     : {int(n_rel)} ({n_rel/total_rows*100:.1f}%)")
    print(f"  Irrelevant   : {int(total_rows - n_rel)} ({(total_rows - n_rel)/total_rows*100:.1f}%)")
    if n_failed:
        print(f"  Failed rows  : {int(n_failed)} — run Section 5 retry cell to fix")
    
    print(f"\n  Category breakdown (relevant only):")
    for cat, count in df[df['relevant'] == True]['category'].value_counts().items():
        print(f"    {cat:<14} {count}")
    
    print(f"\n  Sentiment breakdown (relevant only):")
    for sent, count in df[df['relevant'] == True]['sentiment'].value_counts().items():
        print(f"    {sent:<14} {count}")


df_summary = pd.read_csv("../data/processed/tz_headlines_labelled.csv")
summary(df_summary)


DONE
  Total rows   : 5310
  Relevant     : 3836 (72.2%)
  Irrelevant   : 1474 (27.8%)

  Category breakdown (relevant only):
    Banking        785
    Agriculture    593
    Trade          580
    General        551
    Policy         508
    Investment     420
    Energy         173
    Forex          119
    Transport      101
    Tourism        3
    Mining         2
    Inflation      1

  Sentiment breakdown (relevant only):
    Neutral        2398
    Positive       1114
    Negative       323
